In [ ]:
!pip uninstall -y torchvision
!pip install -U transformers accelerate soundfile librosa pydub

In [ ]:
AUDIO_WAV = "/content/drive/MyDrive/мага/МД/narty_16k.wav"
CLEAN_TEXT_FILE = "/content/drive/MyDrive/мага/МД/clean_ossetian.txt"

In [ ]:
import os

print("Аудио найдено:", os.path.exists(AUDIO_WAV))
print("Текст найден:", os.path.exists(CLEAN_TEXT_FILE))

In [ ]:
with open(CLEAN_TEXT_FILE, "r", encoding="utf-8") as f:
    clean_lines = [line.strip() for line in f if line.strip()]

print("Количество предложений:", len(clean_lines))
print(clean_lines[:5])

In [ ]:
import re

with open(CLEAN_TEXT_FILE, "r", encoding="utf-8-sig") as f:
    full_text = f.read()

# Убираем лишние переносы строк и пробелы
full_text = re.sub(r"\s+", " ", full_text).strip()

# Делим текст на предложения по точке, вопросительному, восклицательному знаку и многоточию
clean_lines = re.findall(r"[^.!?…]+[.!?…]?", full_text)

# Чистим пробелы
clean_lines = [s.strip() for s in clean_lines if s.strip()]

print("Количество предложений:", len(clean_lines))
print(clean_lines[:10])
print(clean_lines[-5:])

In [ ]:
import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC, pipeline

MODEL_ID = "facebook/mms-1b-all"
LANG = "oss"  # осетинский язык

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe_device = 0 if torch.cuda.is_available() else -1

print("Устройство:", device)

print("Загружаю processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, target_lang=LANG)

print("Загружаю модель...")
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID,
    target_lang=LANG,
    ignore_mismatched_sizes=True,
    low_cpu_mem_usage=True
)

model.to(device)

asr = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=pipe_device
)

print("Готово, модель загружена.")

In [ ]:
AUDIO_TEST = "test_3min.wav"

!ffmpeg -y -i "$AUDIO_WAV" -t 00:03:00 -ac 1 -ar 16000 "$AUDIO_TEST"

In [ ]:
import os

print(os.path.exists(AUDIO_TEST))

In [ ]:
print("Распознаю тестовый фрагмент...")

result = asr(
    AUDIO_WAV,
    return_timestamps="word",
    chunk_length_s=20,
    stride_length_s=(2, 2)
)

print("Распознанный текст:")
print(result["text"])

print("\nПервые элементы с таймкодами:")
for chunk in result["chunks"][:20]:
    print(chunk)

In [ ]:
MAX_WORD_PAUSE = 0.7

word_chunks = []

for ch in result["chunks"]:
    text = ch.get("text", "").strip()
    timestamp = ch.get("timestamp")

    if not text or timestamp is None:
        continue

    start, end = timestamp

    if start is None or end is None:
        continue

    word_chunks.append({
        "text": text,
        "start": float(start),
        "end": float(end)
    })

print("Количество слов с таймкодами:", len(word_chunks))
print(word_chunks[:10])

In [ ]:
phrases = []

if word_chunks:
    current_words = [word_chunks[0]["text"]]
    current_start = word_chunks[0]["start"]
    current_end = word_chunks[0]["end"]

    for word in word_chunks[1:]:
        pause = word["start"] - current_end

        if pause <= MAX_WORD_PAUSE:
            current_words.append(word["text"])
            current_end = word["end"]
        else:
            phrases.append({
                "text": " ".join(current_words),
                "start": current_start,
                "end": current_end
            })

            current_words = [word["text"]]
            current_start = word["start"]
            current_end = word["end"]

    phrases.append({
        "text": " ".join(current_words),
        "start": current_start,
        "end": current_end
    })

print("Количество фраз:", len(phrases))

for p in phrases[:20]:
    print(f'{p["text"]} | {p["start"]} | {p["end"]}')

In [ ]:
from difflib import SequenceMatcher
import re
import csv
import os

def normalize_text(text):
    """
    Приводим текст к более простому виду:
    маленькие буквы, без лишних знаков препинания и пробелов.
    Это нужно, чтобы сравнение было мягче.
    """
    text = text.lower()
    text = text.replace("æ", "ӕ").replace("Æ", "ӕ")
    text = re.sub(r"[^\w\sӕӔ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def similarity(a, b):
    """
    Считаем, насколько похожи две строки.
    1.0 — полностью похожи.
    0.0 — совсем не похожи.
    """
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


THRESHOLD = 0.45   # порог похожести. Если плохо сопоставляет, можно снизить до 0.35
LOOKAHEAD = 10     # сколько следующих распознанных фраз смотрим вперёд
MAX_MERGE = 4      # сколько соседних фраз MMS можно склеивать в одну

aligned = []
j_start = 0

for i, clean_text in enumerate(clean_lines, start=1):
    best = None

    # Ищем подходящую распознанную фразу рядом с текущей позицией
    for j in range(j_start, min(len(phrases), j_start + LOOKAHEAD)):
        combined_text = ""
        start_time = phrases[j]["start"]
        end_time = phrases[j]["end"]

        # Иногда одно предложение может быть разбито на несколько фраз,
        # поэтому пробуем склеить несколько соседних распознанных фрагментов
        for k in range(j, min(len(phrases), j + MAX_MERGE)):
            combined_text = (combined_text + " " + phrases[k]["text"]).strip()
            end_time = phrases[k]["end"]

            score = similarity(clean_text, combined_text)

            if best is None or score > best["score"]:
                best = {
                    "clean_id": i,
                    "clean_text": clean_text,
                    "recognized_text": combined_text,
                    "start": start_time,
                    "end": end_time,
                    "score": score,
                    "j_from": j,
                    "j_to": k
                }

    # Если похожесть достаточная, считаем, что нашли соответствие
    if best and best["score"] >= THRESHOLD:
        aligned.append(best)
        j_start = best["j_to"] + 1
    else:
        # Если ничего нормального не нашли, всё равно сохраняем строку,
        # но без таймкодов
        aligned.append({
            "clean_id": i,
            "clean_text": clean_text,
            "recognized_text": "",
            "start": "",
            "end": "",
            "score": 0,
            "j_from": "",
            "j_to": ""
        })

print("Всего предложений в чистом тексте:", len(clean_lines))
print("Всего сопоставленных строк:", len(aligned))

success = sum(1 for row in aligned if row["start"] != "")
print("Успешно сопоставлено:", success)

print("\nПервые 10 сопоставлений:")
for row in aligned:
    print("-----")
    print("Чистый текст:", row["clean_text"])
    print("Распознано:", row["recognized_text"])
    print("Время:", row["start"], "-", row["end"])
    print("Похожесть:", round(row["score"], 3))

In [ ]:
import os
import csv
import shutil
from pydub import AudioSegment

# Папка, куда сохраняем итоговый датасет на Google Диске
DATASET_DIR = "/content/drive/MyDrive/ossetian_dataset_final"
WAVS_DIR = os.path.join(DATASET_DIR, "wavs")

# Создаём папки
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(WAVS_DIR, exist_ok=True)

# Загружаем полный аудиофайл
audio = AudioSegment.from_wav(AUDIO_WAV)

# Файлы с метаданными
METADATA_SIMPLE = os.path.join(DATASET_DIR, "metadata.csv")
METADATA_FULL = os.path.join(DATASET_DIR, "metadata_full.csv")

saved = 0

with open(METADATA_SIMPLE, "w", encoding="utf-8", newline="") as simple_file, \
     open(METADATA_FULL, "w", encoding="utf-8", newline="") as full_file:

    simple_writer = csv.writer(simple_file, delimiter="|")
    full_writer = csv.writer(full_file)

    # Полная таблица с дополнительной информацией
    full_writer.writerow([
        "id",
        "audio_path",
        "clean_text",
        "recognized_text",
        "start_sec",
        "end_sec",
        "duration_sec",
        "similarity"
    ])

    for row in aligned:
        # Пропускаем строки, где не удалось найти таймкоды
        if row["start"] == "":
            continue

        clip_id = f'clip_{row["clean_id"]:06d}'
        filename = f"{clip_id}.wav"
        output_path = os.path.join(WAVS_DIR, filename)

        START_PADDING_MS = 150   # небольшой запас перед фразой
        END_PADDING_MS = 500     # запас после фразы, чтобы не обрезать конец

        start_ms = int(float(row["start"]) * 1000)
        end_ms = int(float(row["end"]) * 1000)

        # Добавляем запас, но не выходим за границы аудио
        start_ms = max(0, start_ms - START_PADDING_MS)
        end_ms = min(len(audio), end_ms + END_PADDING_MS)

        if end_ms <= start_ms:
            continue

        # Вырезаем фрагмент из полного аудио
        chunk = audio[start_ms:end_ms]
        chunk.export(output_path, format="wav")

        duration_sec = round(end_ms - start_ms, 3)

        clean_text = row["clean_text"]
        recognized_text = row["recognized_text"]
        similarity = round(row["score"], 3)

        # Простой формат датасета:
        # имя_файла | правильный текст | правильный текст
        # Такой формат часто используют для TTS
        simple_writer.writerow([
            f"wavs/{filename}",
            clean_text,
            clean_text
        ])

        # Полный формат для проверки качества
        full_writer.writerow([
            clip_id,
            f"wavs/{filename}",
            clean_text,
            recognized_text,
            start_sec,
            end_sec,
            duration_sec,
            similarity
        ])

        saved += 1

print("Готово.")
print("Сохранено фрагментов:", saved)
print("Папка датасета:", DATASET_DIR)
print("Простой metadata:", METADATA_SIMPLE)
print("Полный metadata:", METADATA_FULL)